# 08 – Final holdout (2024–2025)
Run once, after everything else. Trains on all development data and checks the benchmarks, logistic regression and the price-level model on the untouched holdout.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import os
from datetime import datetime

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, matthews_corrcoef,
                             mean_squared_error, mean_absolute_error, r2_score)

from google.colab import drive
drive.mount('/content/drive')

BASE     = "/content/drive/MyDrive/CS5500_dissertation"
FEATURES = f"{BASE}/features"
RESULTS  = f"{BASE}/results"

TICKERS      = ["AAPL", "MSFT", "JPM", "XOM", "SPY"]
DEV_END      = "2023-12-31"
HOLDOUT_FROM = "2024-01-01"
SEED         = 42
TRADING_DAYS = 252

np.random.seed(SEED)

full, dev, hold = {}, {}, {}
for t in TICKERS:
    f = pd.read_csv(f"{FEATURES}/{t}_features.csv", index_col=0, parse_dates=True)
    full[t] = f
    dev[t]  = f.loc[:DEV_END]
    hold[t] = f.loc[HOLDOUT_FROM:]
    print(f"{t}: development {len(dev[t])} rows, holdout {len(hold[t])} rows "
          f"({hold[t].index.min().date()} to {hold[t].index.max().date()})")

PREDICTORS = [c for c in dev["AAPL"].columns
              if not c.startswith("target") and c != "price_today"]

print(f"\n{len(PREDICTORS)} predictors")

## 2. Confirm the boundary is clean
Verify that no holdout row precedes any development row before using either.

In [ ]:
violations = 0
for t in TICKERS:
    last_dev   = dev[t].index.max()
    first_hold = hold[t].index.min()
    ok = last_dev < first_hold
    violations += (not ok)
    print(f"{t}: last development {last_dev.date()} < first holdout "
          f"{first_hold.date()}  ->  {'ok' if ok else 'VIOLATION'}")

print(f"\nViolations: {violations}")
print("PASS - holdout is strictly later than development data" if violations == 0
      else "STOP - boundary is broken, do not proceed")

## 3. Benchmarks on the holdout
The two zero-parameter rules, evaluated on data never previously examined.

In [ ]:
def score(y_true, y_pred, ticker, model):
    acc = accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    se  = np.sqrt(acc * (1 - acc) / len(y_true))
    return {"ticker": ticker, "model": model, "n": len(y_true),
            "accuracy": round(acc, 4),
            "ci_low":   round(acc - 1.96 * se, 4),
            "ci_high":  round(acc + 1.96 * se, 4),
            "mcc":      round(mcc, 4)}


rows = []
for t in TICKERS:
    h = hold[t]
    y = h["target_direction"].values

    rows.append(score(y, np.ones(len(y), dtype=int), t, "majority_class"))
    rows.append(score(y, (h["ret_lag_1"].values > 0).astype(int), t, "persistence"))

benchmarks = pd.DataFrame(rows)
print("Holdout benchmarks, 2024-01-01 onward\n")
print(benchmarks.to_string(index=False))

## 4. Models on the holdout
Trained on all development data through 2023-12-31, then applied once to the holdout.

In [ ]:
model_rows = []
price_rows = []

for t in TICKERS:
    d, h = dev[t], hold[t]

    # ---- directional model: logistic regression, sound protocol ----------
    Xd = d[PREDICTORS].values
    Xh = h[PREDICTORS].values

    scaler = StandardScaler().fit(Xd)          # development only
    clf = LogisticRegression(max_iter=1000, random_state=SEED)
    clf.fit(scaler.transform(Xd), d["target_direction"].values)

    pred = clf.predict(scaler.transform(Xh))
    model_rows.append(score(h["target_direction"].values, pred, t, "logistic_regression"))

    # ---- price-target arm, as used by the audited literature -------------
    def with_price_lags(df, n_lags=5):
        x = df.copy()
        for lag in range(n_lags):
            x[f"px_lag_{lag}"] = x["price_today"].shift(lag)
        return x.dropna()

    dl, hl = with_price_lags(d), with_price_lags(h)
    cols = PREDICTORS + [f"px_lag_{i}" for i in range(5)]

    sc2 = StandardScaler().fit(dl[cols].values)
    reg = LinearRegression().fit(sc2.transform(dl[cols].values),
                                 dl["target_price"].values)

    p     = reg.predict(sc2.transform(hl[cols].values))
    act   = hl["target_price"].values
    today = hl["price_today"].values

    price_rows.append({
        "ticker": t,
        "r2":          round(r2_score(act, p), 6),
        "rmse":        round(np.sqrt(mean_squared_error(act, p)), 4),
        "naive_rmse":  round(np.sqrt(mean_squared_error(act, today)), 4),
        "beats_naive": bool(np.sqrt(mean_squared_error(act, p))
                            < np.sqrt(mean_squared_error(act, today))),
        "dir_acc":     round(accuracy_score(hl["target_direction"].values,
                                            (p > today).astype(int)), 4),
        "mcc":         round(matthews_corrcoef(hl["target_direction"].values,
                                               (p > today).astype(int)), 4),
    })

models = pd.DataFrame(model_rows)
price  = pd.DataFrame(price_rows)

print("Directional model on holdout\n")
print(models.to_string(index=False))
print("\n\nPrice-target arm on holdout\n")
print(price.to_string(index=False))
print(f"\nMean R2: {price.r2.mean():.6f}")
print(f"Mean directional accuracy: {price.dir_acc.mean():.4f}")
print(f"Beats naive on RMSE: {int(price.beats_naive.sum())}/{len(price)}")

## 5. Model against benchmarks

In [ ]:
combined = pd.concat([benchmarks, models])

comparison = combined.pivot(index="ticker", columns="model",
                            values=["accuracy", "mcc"]).round(4)

print("Holdout: all approaches side by side\n")
print(comparison.to_string())

lr  = models.set_index("ticker")["accuracy"]
maj = benchmarks.query("model == 'majority_class'").set_index("ticker")["accuracy"]
per = benchmarks.query("model == 'persistence'").set_index("ticker")["accuracy"]

print(f"\nAssets where the model beats majority class: "
      f"{int((lr > maj).sum())}/{len(lr)}")
print(f"Assets where the model beats persistence:    "
      f"{int((lr > per).sum())}/{len(lr)}")
print(f"\nMean accuracy - model {lr.mean():.4f}, "
      f"majority {maj.mean():.4f}, persistence {per.mean():.4f}")
print(f"Mean MCC - model {models.mcc.mean():.4f}")

## 6. Economic evaluation on the holdout

In [ ]:
def backtest(signal, next_return, cost_bps):
    signal = np.asarray(signal, dtype=float)
    nr     = np.asarray(next_return, dtype=float)
    prev   = np.concatenate([[0.0], signal[:-1]])
    trades = np.abs(signal - prev)

    r = signal * nr - trades * (cost_bps / 10_000)
    eq = np.cumprod(1 + r)
    dd = eq / np.maximum.accumulate(eq) - 1

    ar = r.mean() * TRADING_DAYS
    av = r.std(ddof=1) * np.sqrt(TRADING_DAYS)
    return {"sharpe": ar / av if av > 0 else np.nan,
            "ann_return": ar, "max_drawdown": dd.min(),
            "turnover_pa": trades.sum() / len(signal) * TRADING_DAYS}


econ = []
for t in TICKERS:
    d, h = dev[t], hold[t]

    scaler = StandardScaler().fit(d[PREDICTORS].values)
    clf = LogisticRegression(max_iter=1000, random_state=SEED)
    clf.fit(scaler.transform(d[PREDICTORS].values), d["target_direction"].values)
    sig = clf.predict(scaler.transform(h[PREDICTORS].values))

    nr = h["target_return"].values

    for name, s in [("model", sig),
                    ("persistence", (h["ret_lag_1"].values > 0).astype(int)),
                    ("buy_hold", np.ones(len(h)))]:
        for c in [0, 5, 10, 20]:
            b = backtest(s, nr, c)
            econ.append({"ticker": t, "strategy": name, "cost_bps": c,
                         "sharpe": round(b["sharpe"], 4),
                         "ann_return": round(b["ann_return"], 4),
                         "max_drawdown": round(b["max_drawdown"], 4),
                         "turnover_pa": round(b["turnover_pa"], 1)})

economics = pd.DataFrame(econ)

print("Holdout economic performance, mean across five assets\n")
print(economics.groupby(["strategy", "cost_bps"])
               [["ann_return", "sharpe", "max_drawdown", "turnover_pa"]]
               .mean().round(4).to_string())

## 7. Save everything and log

In [ ]:
benchmarks.to_csv(f"{RESULTS}/holdout_benchmarks.csv", index=False)
models.to_csv(f"{RESULTS}/holdout_models.csv", index=False)
price.to_csv(f"{RESULTS}/holdout_price_target.csv", index=False)
economics.to_csv(f"{RESULTS}/holdout_economics.csv", index=False)

today_str = datetime.now().strftime("%Y-%m-%d")

lines = []
lines.append("")
lines.append("---")
lines.append("")
lines.append(f"## {today_str} - Notebook 08: FINAL HOLDOUT EVALUATION (run once)")
lines.append("")
lines.append(f"Development period: through {DEV_END}")
lines.append(f"Holdout period: {HOLDOUT_FROM} onward, n = {len(hold['AAPL'])} per asset")
lines.append(f"Boundary check: {violations} violations")
lines.append("Models trained on all development data; scaler fitted on development only.")
lines.append("No hyperparameter was adjusted for this evaluation.")
lines.append("")
lines.append("**Benchmarks**")
lines.append("")
lines.append(benchmarks.to_string(index=False))
lines.append("")
lines.append("**Directional model**")
lines.append("")
lines.append(models.to_string(index=False))
lines.append("")
lines.append("**Price-target arm**")
lines.append("")
lines.append(price.to_string(index=False))
lines.append("")
lines.append(f"- Mean R2: {price.r2.mean():.6f}")
lines.append(f"- Mean directional accuracy: {price.dir_acc.mean():.4f}")
lines.append(f"- Beats naive on RMSE: {int(price.beats_naive.sum())}/{len(price)}")
lines.append("")
lines.append("**Model versus benchmarks**")
lines.append(f"- Beats majority class on {int((lr > maj).sum())}/{len(lr)} assets")
lines.append(f"- Beats persistence on {int((lr > per).sum())}/{len(lr)} assets")
lines.append(f"- Mean accuracy: model {lr.mean():.4f}, majority {maj.mean():.4f}, "
             f"persistence {per.mean():.4f}")
lines.append(f"- Mean MCC: {models.mcc.mean():.4f}")
lines.append("")
lines.append("**Economic performance**")
lines.append("")
lines.append(economics.groupby(["strategy", "cost_bps"])
                      [["ann_return", "sharpe", "max_drawdown", "turnover_pa"]]
                      .mean().round(4).to_string())
lines.append("")
lines.append("THIS EVALUATION IS FINAL. No further tuning is permitted on these data.")

entry = "\n".join(lines)

with open(f"{BASE}/results_log.md", "a") as f:
    f.write(entry)

print(entry)